# Multi-agent systems

A second agent is useful only when it has a clear job that should stay separate.

In this notebook, we will:

- turn a specialist agent into a tool;
- give that specialist fewer permissions;
- keep worker context small;
- hand a conversation to another agent; and
- compare the extra cost with a single agent.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. Make a policy specialist

The specialist can read orders and policies. It cannot move money or make the final decision. This is least privilege: give a role only the tools it needs.

In [ ]:
from shoplab.loop import run_agent
from shoplab.tools import Tool, standard_tools
from shoplab.world import load_tickets

def only(*names):
    all_tools = standard_tools()
    return {name: all_tools[name] for name in names}

RESEARCHER = (
    "You are a policy researcher. Use the order and policy tools. "
    "Return a short finding with the policy id and any amount. "
    "You do not make the final decision."
)

def research(question):
    result = run_agent(
        question,
        only("get_order", "search_policy"),
        model=MODEL,
        system=RESEARCHER,
        max_steps=4,
    )
    return {"finding": result.answer, "stop_reason": result.stop_reason}

research_tool = Tool(
    name="research_policy",
    description="Ask a policy specialist one focused question.",
    params={
        "type": "object",
        "properties": {"question": {"type": "string"}},
        "required": ["question"],
    },
    fn=research,
)

print(research_tool.fn("For ORD-7312, what policy applies to opened boots returned in time?"))

## 2. Let a supervisor call the specialist

The supervisor sees the specialist as one tool. The worker's internal messages stay inside the worker.

In [ ]:
ticket = next(
    item
    for item in load_tickets()["train"]
    if item["ticket_id"] == "TKT-2205"
)
task = json.dumps({key: value for key, value in ticket.items() if key != "gold"})

SUPERVISOR = (
    "You supervise return decisions. Ask research_policy one complete question. "
    "Then call finish with decision, policy_id, and refund_usd."
)

supervisor_tools = {
    "research_policy": research_tool,
    "finish": standard_tools()["finish"],
}

supervisor_result = run_agent(
    task,
    supervisor_tools,
    model=MODEL,
    system=SUPERVISOR,
    max_steps=4,
)

print("answer:", supervisor_result.answer)
print("known:", ticket["gold"])

A trace would show research_policy as a nested agent call. That nesting is useful when we need to see which agent spent time and tokens.

## 3. Give the worker focused context

A worker usually needs one focused question, not the supervisor's entire history. Smaller context is cheaper and reduces distraction.

In [ ]:
from shoplab.context import count_tokens
from shoplab import world

focused = "For ORD-7312, which policy applies to opened boots returned in time?"
shared = (
    task
    + "\nOrder data: " + json.dumps(next(item for item in world.load_orders() if item["order_id"] == "ORD-7312"))
    + "\nSeveral policy results: " + json.dumps(world.search_policy("opened refund", k=5))
    + "\nQuestion: " + focused
)

print("focused tokens:", count_tokens(focused, model=MODEL))
print("shared tokens:", count_tokens(shared, model=MODEL))

## 4. Handoff is different from calling a worker

A worker answers one request and returns control. In a handoff, the next agent takes ownership of the conversation.

In [ ]:
risk_ticket = next(
    item
    for item in load_tickets()["train"]
    if item["ticket_id"] == "TKT-2209"
)
risk_task = json.dumps({key: value for key, value in risk_ticket.items() if key != "gold"})

TRIAGE = (
    "Check the order and customer. If you see fraud or abuse risk, reply HANDOFF. "
    "Otherwise call finish."
)

triage_result = run_agent(
    risk_task,
    only("get_order", "get_customer", "search_policy", "finish"),
    model=MODEL,
    system=TRIAGE,
    max_steps=5,
)

print("triage result:", triage_result.answer)

In [ ]:
conversation = json.dumps(triage_result.messages, default=str)

ESCALATION = (
    "You own this handed-off case. Review the conversation, call escalate, "
    "then call finish with decision escalate and policy_id pol-fraud."
)

escalation_result = run_agent(
    "Ticket " + risk_ticket["ticket_id"] + "\nConversation:\n" + conversation,
    only("get_order", "get_customer", "escalate", "finish"),
    model=MODEL,
    system=ESCALATION,
    max_steps=5,
)

print("new owner:", escalation_result.answer)
print("known:", risk_ticket["gold"])

## 5. Measure the extra agent

A supervisor and worker add another model boundary. That means more calls, more tokens, and another place where details can be lost.

In [ ]:
from shoplab.llm import LEDGER

SINGLE = (
    "Handle the return. Look up order, customer, and policy. "
    "Then call finish with decision, policy_id, and refund_usd."
)

def measure(run):
    start = len(LEDGER)
    answer = run()
    rows = LEDGER[start:]
    return {
        "calls": len(rows),
        "tokens": sum(row["prompt_tokens"] + row["completion_tokens"] for row in rows),
        "answer": answer,
    }

single = measure(lambda: run_agent(
    task, standard_tools(), model=MODEL, system=SINGLE, max_steps=8
).answer)
multi = measure(lambda: run_agent(
    task, supervisor_tools, model=MODEL, system=SUPERVISOR, max_steps=4
).answer)

print("single:", single)
print("multi:", multi)

Use another agent when it needs a different toolset, private context, or a clear ownership boundary. Do not add one only because the org chart looks tidy.

## What to remember

1. More agents are not automatically better.
2. An agent can be wrapped as a tool.
3. Specialists should receive only the tools they need.
4. Focused worker context saves tokens.
5. A handoff changes who owns the loop.
6. Measure a multi-agent design against a simpler single agent.

## Small practice steps

1. Remove get_order from the researcher's tools and observe the limit.
2. Add more history to shared and count the tokens again.
3. Compare the single and multi answers field by field.